In [6]:
import os
import csv
import re
import nltk
import pandas as pd
import openpyxl
from nltk import bigrams, word_tokenize
from collections import defaultdict

nltk.download('punkt')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\pc\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\pc\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\pc\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
def procurar_e_imprimir(textos, termos):
    resultados = {termo: [] for termo in termos}
    textos_utilizados = set()

    for termo in termos:
        bigramas_termo = list(bigrams(termo.lower().split()))

        for texto in textos:
            if texto['text'] in textos_utilizados:
                continue

            texto_lower = texto['text'].lower()
            palavras_texto = word_tokenize(texto_lower)
            bigramas_texto = list(bigrams(palavras_texto))
            count = sum(1 for bigrama in bigramas_termo if bigrama in bigramas_texto)

            if count >= 4:
                texto['correspondencia'] = count
                resultados[termo].append(texto)
                textos_utilizados.add(texto['text'])

    return resultados

def carregar_termos_xlsx(caminho_arquivo):
    wb = openpyxl.load_workbook(caminho_arquivo)
    planilha = wb.active
    termos = [celula.value for celula in planilha['D'] if celula.value]
    return termos

def carregar_textos_csv(caminho_csv):
    textos = []
    encodings = ['utf-8', 'latin1', 'iso-8859-1', 'cp1252']

    for encoding in encodings:
        try:
            with open(caminho_csv, 'r', encoding=encoding, errors='ignore') as arquivo:
                leitor_csv = csv.DictReader(arquivo)
                for row in leitor_csv:
                    if 'text' in row and row['text']:
                        textos.append({key: row[key] for key in row})
            return textos
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"Erro inesperado ao ler '{caminho_csv}': {e}")
            break

    return textos

def encontrar_arquivos(diretorio, regex):
    arquivos_encontrados = []
    for root, _, files in os.walk(diretorio):
        for file in files:
            if re.match(regex, file):
                arquivos_encontrados.append(os.path.join(root, file))
    return arquivos_encontrados

def agrupar_por_nome_base(arquivos_csv):
    agrupados = defaultdict(list)
    for caminho_csv in arquivos_csv:
        nome_base = os.path.basename(caminho_csv).split('_cluster_')[0]
        agrupados[nome_base].append(caminho_csv)
    return agrupados

def main():
    diretorio_base = 'C://Users//pc//Desktop//Pdpd_2023//Rotulados_termos'

    # Encontrar subpastas de cada mês (jan_2022, fev_2022, etc.)
    pastas_mensais = [os.path.join(diretorio_base, pasta) for pasta in os.listdir(diretorio_base) 
                      if os.path.isdir(os.path.join(diretorio_base, pasta)) and re.match(r'[a-zA-Z]{3}_2022', pasta)]
    
    arquivos_csv = []
    for pasta_mes in pastas_mensais:
        # Dentro de cada pasta de mês, procurar pelas subpastas 'cluster_rts'
        pasta_cluster_rts = os.path.join(pasta_mes, 'clusters_RTs')
        if os.path.isdir(pasta_cluster_rts):
            # Procurar pelos arquivos CSV dentro da subpasta 'cluster_rts'
            arquivos_csv.extend(encontrar_arquivos(pasta_cluster_rts, r'[A-Za-z]{3}_2022_cluster_\d+\.csv'))

    if not arquivos_csv:
        print("Nenhum arquivo CSV encontrado com o padrão clusters_RTs.")
        return

    # Carregar os termos do primeiro arquivo .xlsx encontrado
    arquivos_xlsx = encontrar_arquivos(diretorio_base, r'.*\.xlsx$')
    if not arquivos_xlsx:
        print("Nenhum arquivo .xlsx encontrado.")
        return

    termos = carregar_termos_xlsx(arquivos_xlsx[0])

    arquivos_agrupados = agrupar_por_nome_base(arquivos_csv)

    for nome_base, caminhos_csv in arquivos_agrupados.items():
        resultados_por_cluster = defaultdict(list)

        for caminho_csv in caminhos_csv:
            cluster_nome = os.path.basename(caminho_csv).split('_cluster_')[-1].replace('.csv', '')
            textos = carregar_textos_csv(caminho_csv)

            if not textos:
                print(f"Arquivo {caminho_csv} está vazio ou não contém a coluna 'text'.")
                continue

            resultados = procurar_e_imprimir(textos, termos)

            for termo, textos_relevantes in resultados.items():
                for texto in textos_relevantes:
                    texto['termo'] = termo
                    texto['cluster'] = cluster_nome
                    resultados_por_cluster[cluster_nome].append(texto)

        # Salvar os resultados em um arquivo Excel
        caminho_resultado = os.path.join(diretorio_base, f'{nome_base}_resultados.xlsx')
        with pd.ExcelWriter(caminho_resultado) as writer:
            if resultados_por_cluster:
                for cluster, textos in resultados_por_cluster.items():
                    if textos:
                        df_resultados = pd.DataFrame(textos)
                        df_resultados.to_excel(writer, sheet_name=f'Cluster_{cluster}', index=False)
            else:
                pd.DataFrame().to_excel(writer, sheet_name='Sem Resultados')

        print(f'Resultados para {nome_base} salvos com sucesso em {caminho_resultado}')

if __name__ == "__main__":
    main()
